In [10]:
import requests

url = "https://api.open-meteo.com/v1/forecast"
params = {
    "latitude": -7.1167,
    "longitude": 112.4167,
    "hourly": "temperature_2m,relative_humidity_2m,precipitation",
    "timezone": "Asia/Jakarta",
}

response = requests.get(url, params=params, timeout=30)
response.raise_for_status() 
data = response.json()

print(data.keys())
print(data["hourly"].keys())
print(data["hourly"]["time"][:3])

dict_keys(['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'hourly_units', 'hourly'])
dict_keys(['time', 'temperature_2m', 'relative_humidity_2m', 'precipitation'])
['2026-09-28T00:00', '2026-09-28T01:00', '2026-09-28T02:00']


In [11]:
# %pip install openmeteo-requests
# %pip install requests-cache retry-requests numpy pandas

In [12]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://api.open-meteo.com/v1/forecast"
params = {
	"latitude": -7.1167,
	"longitude": 112.4167,
	"daily": ["temperature_2m_max", "temperature_2m_min", "precipitation_sum", "precipitation_hours", "wind_speed_10m_max", "wind_gusts_10m_max", "shortwave_radiation_sum", "temperature_2m_mean", "relative_humidity_2m_mean"],
	"hourly": "weather_code",
}
responses = openmeteo.weather_api(url, params = params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_weather_code = hourly.Variables(0).ValuesAsNumpy()

hourly_data = {
	"date": pd.date_range(
		start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
		end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = hourly.Interval()),
		inclusive = "left"
	)
}

hourly_data["weather_code"] = hourly_weather_code

hourly_dataframe = pd.DataFrame(data = hourly_data)
print("\nHourly data\n", hourly_dataframe)

# Process daily data. The order of variables needs to be the same as requested.
daily = response.Daily()
daily_temperature_2m_max = daily.Variables(0).ValuesAsNumpy()
daily_temperature_2m_min = daily.Variables(1).ValuesAsNumpy()
daily_precipitation_sum = daily.Variables(2).ValuesAsNumpy()
daily_precipitation_hours = daily.Variables(3).ValuesAsNumpy()
daily_wind_speed_10m_max = daily.Variables(4).ValuesAsNumpy()
daily_wind_gusts_10m_max = daily.Variables(5).ValuesAsNumpy()
daily_shortwave_radiation_sum = daily.Variables(6).ValuesAsNumpy()
daily_temperature_2m_mean = daily.Variables(7).ValuesAsNumpy()
daily_relative_humidity_2m_mean = daily.Variables(8).ValuesAsNumpy()

daily_data = {
	"date": pd.date_range(
		start = pd.to_datetime(daily.Time(), unit = "s", utc = True),
		end =  pd.to_datetime(daily.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = daily.Interval()),
		inclusive = "left"
	)
}

daily_data["temperature_2m_max"] = daily_temperature_2m_max
daily_data["temperature_2m_min"] = daily_temperature_2m_min
daily_data["precipitation_sum"] = daily_precipitation_sum
daily_data["precipitation_hours"] = daily_precipitation_hours
daily_data["wind_speed_10m_max"] = daily_wind_speed_10m_max
daily_data["wind_gusts_10m_max"] = daily_wind_gusts_10m_max
daily_data["shortwave_radiation_sum"] = daily_shortwave_radiation_sum
daily_data["temperature_2m_mean"] = daily_temperature_2m_mean
daily_data["relative_humidity_2m_mean"] = daily_relative_humidity_2m_mean

daily_dataframe = pd.DataFrame(data = daily_data)
print("\nDaily data\n", daily_dataframe)

Coordinates: -7.135324954986572°N 112.44293975830078°E
Elevation: 6.0 m asl
Timezone difference to GMT+0: 0s

Hourly data
                          date  weather_code
0   2026-09-28 00:00:00+00:00           1.0
1   2026-09-28 01:00:00+00:00           0.0
2   2026-09-28 02:00:00+00:00           0.0
3   2026-09-28 03:00:00+00:00           0.0
4   2026-09-28 04:00:00+00:00           0.0
..                        ...           ...
163 2026-10-04 19:00:00+00:00           3.0
164 2026-10-04 20:00:00+00:00           2.0
165 2026-10-04 21:00:00+00:00           2.0
166 2026-10-04 22:00:00+00:00           1.0
167 2026-10-04 23:00:00+00:00           0.0

[168 rows x 2 columns]

Daily data
                        date  temperature_2m_max  temperature_2m_min  \
0 2026-09-28 00:00:00+00:00           34.549999           25.000000   
1 2026-09-29 00:00:00+00:00           34.650002           23.799999   
2 2026-09-30 00:00:00+00:00           34.750000           24.750000   
3 2026-10-01 00:00:00+00:00 

In [13]:
import openmeteo_requests
import pandas as pd
import requests_cache
from retry_requests import retry

# ---------- Konfigurasi ----------
LAT, LON = -7.1167, 112.4167          # pusat kota Lamongan
START_DATE = "2010-01-01"
END_DATE = "2026-09-20"               # beberapa hari sebelum hari ini (arsip tertinggal)
TIMEZONE = "Asia/Jakarta"
OUTPUT_CSV = "cuaca_lamongan_harian.csv"

# Urutan di sini = urutan indeks di response.Daily().Variables(i)
DAILY_VARS = [
    "weather_code",
    "temperature_2m_max",
    "temperature_2m_min",
    "temperature_2m_mean",
    "relative_humidity_2m_mean",
    "precipitation_sum",
    "precipitation_hours",
    "wind_speed_10m_max",
    "wind_gusts_10m_max",
    "shortwave_radiation_sum",
]

# Nama kolom disamakan dengan CSV sebelumnya
RENAME = {
    "date": "tanggal",
    "weather_code": "kode_cuaca",
    "temperature_2m_max": "suhu_maks (°C)",
    "temperature_2m_min": "suhu_min (°C)",
    "temperature_2m_mean": "suhu_rata (°C)",
    "relative_humidity_2m_mean": "kelembapan_rata (%)",
    "precipitation_sum": "curah_hujan (mm)",
    "precipitation_hours": "durasi_hujan (jam)",
    "wind_speed_10m_max": "kecepatan_angin_maks (km/jam)",
    "wind_gusts_10m_max": "hembusan_angin_maks (km/jam)",
    "shortwave_radiation_sum": "radiasi_matahari (MJ/m²)",
}

# Pemetaan kode WMO -> label Indonesia (periksa lagi sesuai kebutuhanmu)
WMO_LABEL = {
    0: "Cerah", 1: "Cerah Berawan", 2: "Berawan", 3: "Mendung",
    51: "Gerimis Ringan", 53: "Gerimis Sedang", 55: "Gerimis Lebat",
    61: "Hujan Ringan", 63: "Hujan Sedang", 65: "Hujan Lebat",
}


def fetch_daily() -> pd.DataFrame:
    # Client dengan cache dan retry otomatis
    cache_session = requests_cache.CachedSession(".cache", expire_after=3600)
    retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
    client = openmeteo_requests.Client(session=retry_session)

    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": LAT,
        "longitude": LON,
        "start_date": START_DATE,
        "end_date": END_DATE,
        "timezone": TIMEZONE,
        "daily": DAILY_VARS,
    }
    response = client.weather_api(url, params=params)[0]

    print(f"Koordinat grid: {response.Latitude()}°N {response.Longitude()}°E")
    print(f"Elevasi: {response.Elevation()} m")
    print(f"Selisih zona waktu ke GMT+0: {response.UtcOffsetSeconds()} detik")

    daily = response.Daily()

    # Tanggal: epoch UTC -> waktu Jakarta (jam 00:00 lokal)
    dates = pd.date_range(
        start=pd.to_datetime(daily.Time(), unit="s", utc=True),
        end=pd.to_datetime(daily.TimeEnd(), unit="s", utc=True),
        freq=pd.Timedelta(seconds=daily.Interval()),
        inclusive="left",
    ).tz_convert(TIMEZONE)

    data = {"date": dates.tz_localize(None)}  # buang info zona, sisakan tanggal lokal

    # Loop dengan indeks otomatis: tidak ada penugasan manual yang bisa tertukar
    for i, name in enumerate(DAILY_VARS):
        data[name] = daily.Variables(i).ValuesAsNumpy()

    return pd.DataFrame(data)


def main() -> None:
    df = fetch_daily()

    # Kode cuaca -> bilangan bulat + label teks
    df["weather_code"] = df["weather_code"].astype("Int64")
    df.insert(2, "cuaca", df["weather_code"].map(WMO_LABEL).fillna("Lainnya"))

    df = df.rename(columns=RENAME)

    print("\nBentuk data:", df.shape)
    print("\nNilai kosong per kolom:\n", df.isna().sum())
    print("\n5 baris awal:\n", df.head())
    print("\n5 baris akhir:\n", df.tail())
    print("\nKode cuaca yang tidak dikenal:",
          sorted(df.loc[df["cuaca"] == "Lainnya", "kode_cuaca"].dropna().unique()))

    df.to_csv(OUTPUT_CSV, index=False)
    print(f"\nTersimpan ke {OUTPUT_CSV}")


if __name__ == "__main__":
    main()

Koordinat grid: -7.135324954986572°N 112.44293975830078°E
Elevasi: 6.0 m
Selisih zona waktu ke GMT+0: 25200 detik

Bentuk data: (6107, 12)

Nilai kosong per kolom:
 tanggal                          0
kode_cuaca                       0
cuaca                            0
suhu_maks (°C)                   0
suhu_min (°C)                    0
suhu_rata (°C)                   0
kelembapan_rata (%)              0
curah_hujan (mm)                 0
durasi_hujan (jam)               0
kecepatan_angin_maks (km/jam)    0
hembusan_angin_maks (km/jam)     0
radiasi_matahari (MJ/m²)         0
dtype: int64

5 baris awal:
      tanggal  kode_cuaca           cuaca  suhu_maks (°C)  suhu_min (°C)  \
0 2010-01-01          63    Hujan Sedang       29.569500      24.869501   
1 2010-01-02          53  Gerimis Sedang       30.469501      25.069500   
2 2010-01-03          53  Gerimis Sedang       31.419500      25.319500   
3 2010-01-04          63    Hujan Sedang       30.869501      25.469501   
4 2010-01-0